3

In [7]:
import csv
import heapq
import os

# ── 클래스 정의 ──────────────────────────────────────────
class BirthdayHeap:
    def __init__(self):
        self.heap = []

    def build_heap_from_csv(self, file_path):
        """CSV 파일을 읽어 최대 힙을 구성합니다."""
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"{file_path} 파일이 존재하지 않습니다.")

        with open(file_path, mode='r', encoding='cp949') as f:
            reader = csv.reader(f)
            next(reader)  # 헤더 스킵

            for row in reader:
                if len(row) >= 4 and row[1].strip():
                    try:
                        name = row[0]
                        year  = int(float(row[1]))
                        month = int(float(row[2]))
                        day   = int(float(row[3]))

                        birth_value = year * 10000 + month * 100 + day
                        # 생일이 느린(숫자가 큰) 순서 → 최대 힙 → 음수 부호
                        heapq.heappush(self.heap, (-birth_value, name, year, month, day))
                    except (ValueError, IndexError):
                        continue

    def get_top_n(self, n):
        """힙에서 생일이 느린 순서로 n명을 추출합니다."""
        results = []
        for _ in range(min(n, len(self.heap))):
            neg_val, name, y, m, d = heapq.heappop(self.heap)
            results.append((name, y, m, d))
        return results

# ── 실행 ─────────────────────────────────────────────────
csv_path = os.path.join(os.getcwd(), 'birthday.csv')

bh = BirthdayHeap()
try:
    bh.build_heap_from_csv(csv_path)
except Exception as e:
    print(f"오류 발생: {e}")
else:
    top_10 = bh.get_top_n(10)
    print("생일이 느린 순서 상위 10명")
    print("-" * 35)
    for rank, (name, y, m, d) in enumerate(top_10, start=1):
        print(f"{rank:2}위  {name}  {y}.{m:02d}.{d:02d}")

생일이 느린 순서 상위 10명
-----------------------------------
 1위  이윤서  2006.12.27
 2위  정희원  2006.12.21
 3위  김효린  2006.12.16
 4위  이예은  2006.12.09
 5위  김주영  2006.11.20
 6위  전은빈  2006.11.07
 7위  이하연  2006.09.22
 8위  김우현  2006.09.01
 9위  최윤지  2006.08.30
10위  유가현  2006.08.22


4

In [8]:
import csv
import os

# [1] 노드 클래스 정의 (외부 모듈 없이 직접 정의)
class BidirectNode:
    def __init__(self, x, prev, next):
        self.item = x
        self.prev = prev
        self.next = next

# [2] 원형 이중 연결 리스트 클래스 정의
class CircularDoublyLinkedList:
    def __init__(self):
        self.__head = BidirectNode("dummy", None, None)
        self.__head.prev = self.__head
        self.__head.next = self.__head
        self.__numItems = 0

    def append(self, newItem) -> None:
        prev = self.__head.prev
        newNode = BidirectNode(newItem, prev, self.__head)
        prev.next = newNode
        self.__head.prev = newNode
        self.__numItems += 1

    def getNode(self, i: int) -> BidirectNode:
        curr = self.__head
        for index in range(i + 1):
            curr = curr.next
        return curr

    def __iter__(self):
        return CircularDoublyLinkedListIterator(self)

class CircularDoublyLinkedListIterator:
    def __init__(self, alist):
        self.__head = alist.getNode(-1)
        self.iterPosition = self.__head.next
    def __next__(self):
        if self.iterPosition == self.__head:
            raise StopIteration
        else:
            item = self.iterPosition.item
            self.iterPosition = self.iterPosition.next
            return item

# [3] 데이터 처리 로직
def main():
    input_file = os.path.join(os.getcwd(), 'birthday.csv')
    my_group = "2"  # 2조

    my_list = CircularDoublyLinkedList()

    try:
        with open(input_file, mode='r', encoding='cp949') as f:
            reader = csv.reader(f)
            next(reader)  # 헤더 건너뛰기

            for row in reader:
                try:
                    name  = row[0].strip()
                    group = row[4].strip() if len(row) >= 5 else ""

                    if group != my_group:
                        continue

                    # 생년월일 예외처리: 비어있으면 "생년월일 없음"
                    if len(row) >= 4 and row[1].strip():
                        birth = f"{int(float(row[1]))}.{int(float(row[2])):02d}.{int(float(row[3])):02d}"
                    else:
                        birth = "생년월일 없음"

                    my_list.append([name, birth])

                except Exception:
                    continue

    except FileNotFoundError:
        print(f"❌ 오류: '{input_file}' 파일을 찾을 수 없습니다.")
        return

    # 2조 친구들 출력
    print(f"[{my_group}조 친구들 이름 및 생년월일]")
    print("-" * 35)
    for data in my_list:
        print(f"이름: {data[0]},  생년월일: {data[1]}")

main()

[2조 친구들 이름 및 생년월일]
-----------------------------------
이름: 이채윤,  생년월일: 2005.08.21
이름: 구수진,  생년월일: 2005.04.08
이름: 권지영,  생년월일: 2003.02.05
이름: 김수민,  생년월일: 2006.07.09
이름: 김주영,  생년월일: 2006.11.20
이름: 김태연,  생년월일: 2006.03.22
이름: 김효린,  생년월일: 2006.12.16
이름: 서지윤,  생년월일: 2005.09.22
이름: 유지민,  생년월일: 생년월일 없음
이름: 이승주,  생년월일: 2006.03.26


5

(1) 최대 힙에서 가장 큰 원소가 항상 루트에 있다. 즉, 루트의 깊이가 가장 얕다. 임의의
최대 힙에서 더 얕은 곳에 있는 원소가 더 깊은 곳에 있는 원소보다 더 작은 값을 가질 수
있는가?
-> 없다(최대힙이기 때문에 같을 수는 있으나 더 작은 값을 가질 수는 없다.)

(2) 최대 힙 A[0…n-1]에서 A[0]는 항상 가장 큰 값을 갖고 있다. 반대로 마지막 원소인 A[n1]은 항상 가장 작은 값을 갖는가?
-> 아니다(항상 같거나 작은 값을 갖음)

(3) 임의의 배열 A[0…n-1]을 힙으로 만드는 buildHeap() 알고리즘에서 총 n개의 원소 중
루트의 자격으로 스며내리기를 할지 알아보지 않고 그냥 넘어가는 원소(힙의 노드) 수는
정확하게 몇 개인가?
-> n/2(개) buildHeap()은 자식이 있는 노드 중 가장 뒤에 있는 노드부터 루트까지 거꾸로 올라가며 수행합니다. 따라서 자식이 없어 스며내리기를 할 필요가 없는 바닥 층의 n/2개 노드는 연산 없이 그냥 지나치게 된다.

(4) 배열 A[0…n-1]을 대상으로 하는 스며내리기 알고리즘에서 최악의 경우에 해당하는
시간과 최선의 경우에 해당하는 시간은 어떻게 되는가? Θ −표기로 나타내시오
-> 최악의 경우 Θ(n), 최선의 경우 Θ(1)

(5) 힙인 상태에서 원소 삭제는 루트 노드만 대상으로 한다. 다른 경우는 존재하지 않는다.
테스트 목적으로 힙의 맨 마지막 원소를 삭제하는 작업을 요구한다면 간단한 일인가?
-> 그렇다. 힙의 맨 마지막 원소 삭제는 자식 노드가 없는 말단 노드를 제거하는 것이므로, 트리 구조를 유지하기 위한 별도의 heapify 과정이 필요 없어 매우 간단하다.

(6)어떤 학생이 다음과 같은 질문을 했다. “buildHeap() 알고리즘에서는 아래쪽에서부터
시작해서 스며내리기를 반복하는데 만약 반대 방향으로 하면 어떤가요?(즉, 위쪽에서부터
시작해서 스며오르기를 반복)” 이렇게 해도 힙이 만들어진다. 이 방법은 본문에 제시한
방법에 비해 더 효율적인가? 점근적 시간으로 말하시오.
-> 본문의 방식 (아래에서 위로, 스며내리기): O(n) 학생의 제안 (위에서 아래로, 스며오르기): O(n \log n)
스며내리기의 경우 트리의 아래쪽(리프 노드 근처)에는 노드가 아주 많지만, 이들은 이미 바닥에 있기 때문에 처리할 작업이 거의 없다. 작업량이 많은 루트 노드는 단 하나뿐이다.반면 스며오르기 방식은 노드를 하나씩 추가할 때마다 맨 위까지 올라갈 수 있는지 확인하다. 트리의 깊이가 깊어질수록(아래로 갈수록) 노드의 수는 기하급수적으로 늘어나는데, 이 많은 노드들이 각각 최대 트리 높이(log n)만큼 위로 이동해야 할 가능성이 생긴다. 따라서 학생의 방법으로도 힙을 만들 수는 있지만, 데이터의 개수가 많아질수록 본문에 제시된 아래에서부터 스며내리기를 반복하는 방식이 수학적으로 훨씬 효율적이다.

(7)n개의 원소로 이루어진 최대 힙에서 임의의 원소 값이 증가했을 때 O(log n) 시간에
이를 반영해서 힙을 수선할 수 있다. 어떻게 하면 되는지 그 과정을 기술하시오.
->힙이 이미 구성된 상태에서 특정 원소의 값이 커졌을 때는 자식 노드들과 비교할 필요 없이 자신의 부모 노드와만 비교하며 루트 방향으로 타고 올라가는 '스며오르기' 과정을 통해 힙을 수선할 수 있다. 이 과정은 트리의 가장 밑바닥에서 꼭대기까지 올라가는 최악의 경우를 가정하더라도 트리의 높이인 log n만큼만 이동하면 끝이 나므로, 방대한 데이터 환경에서도 O(log n)이라는 매우 빠른 시간 안에 반영이 가능하다.